# Занятие 4, пара 1. API. JSON и запросы

Автор ноутбука - Дуркин Анатолий Альбертович

Старший преподаватель кафедры прикладной математики и компьютерных наук СГУ им. Питирима Сорокина

Замечания, предложения, идеи, вопросы, связь с автором:
- anatoliy.durkin@mail.ru
- Telegram - @AnatoDu

Больше информации и материалов на канале автора: https://t.me/code_matan_ai

## Сегодня забираем данные по понятным правилам

В прошлый раз мы искали нужные элементы внутри HTML. Но иногда сервис уже умеет отдавать данные программе: отдельно название, отдельно город, отдельно цену. Такая договорённость между программами называется **API** — интерфейсом программирования приложений. Само слово не обещает ни бесплатный доступ, ни хороший набор данных.

Сегодня пройдём путь «текст JSON → словари и списки → аккуратная таблица → осторожный вывод». Научимся читать вложенные поля, замечать пропуски и собирать несколько страниц. Понадобятся списки, словари, цикл `for` и немного Pandas; нужные действия повторим на маленьких примерах.

Основная работа идёт на **придуманных учебных данных**, уже лежащих в комплекте. Это модель структуры API, а не свежие вакансии и не сохранённый ответ конкретного сайта. Интернет и регистрация не нужны. Отдельно посмотрим, где в реальном запросе появляются адрес, статус и ограничения доступа.

Результат пары: таблица восьми учебных записей с происхождением, три упражнения и вывод о том, какие числа вообще можно сравнивать.


### Подключим учебные файлы

Следующая ячейка — служебная настройка комплекта. Она находит папку `course_support` у текущей папки или выше и подключает данные и проверки. Её не нужно писать по памяти или менять для задания. `DATA_DIR` — путь к учебным данным, `WORKSPACE_DIR` — корень вашей рабочей папки. `check(номер, ответы)` сравнивает ответ с условиями упражнения, но не выставляет оценку.

Работайте в своей копии и запускайте ячейки сверху вниз. Переменные из другой тетради не нужны. Если поддержку не удалось найти, откройте тетрадь внутри полного комплекта, а не переносите один файл отдельно. Git и интернет для основной работы не нужны.


In [ ]:
import sys
from pathlib import Path

# Одинаковый запуск в репозитории курса и в личном проекте.
for WORKSPACE_DIR in (Path.cwd(), *Path.cwd().parents):
    if (WORKSPACE_DIR / 'course_support' / '__init__.py').is_file():
        break
else:
    raise FileNotFoundError('Откройте тетрадь внутри папки с course_support; нужен весь комплект проекта')

# При переключении курса в том же kernel не оставляем чужую поддержку в памяти.
loaded = sys.modules.get('course_support')
expected = WORKSPACE_DIR / 'course_support' / '__init__.py'
if loaded is not None and Path(loaded.__file__).resolve() != expected.resolve():
    for name in list(sys.modules):
        if name == 'course_support' or name.startswith('course_support.'):
            del sys.modules[name]
sys.path.insert(0, str(WORKSPACE_DIR))
from course_support import SUPPORT_DIR, DATA_DIR
COURSE_DIR = SUPPORT_DIR
# Только упражнения о личном README: в репозитории курса это ещё чистый шаблон.
PROJECT_DIR = WORKSPACE_DIR / 'project_template' if (WORKSPACE_DIR / 'project_template').is_dir() else WORKSPACE_DIR
from course_support.checks import check

PROJECT_DIR


### 1. JSON — пока просто текст

`json` — модуль стандартной библиотеки Python, устанавливать его не нужно. JSON — текстовый формат обмена данными. Имена полей и строки записываются в двойных кавычках; числа — без кавычек. Внешне похоже на словарь Python, но это ещё не словарь. `type` поможет это увидеть.


In [ ]:
import json
import pandas as pd

text = '{"name": "Стажёр", "salary": 60000, "remote": true, "experience": null}'
print(type(text))
print(text)


`json.loads` разбирает **строку** и возвращает объект Python. Буква `s` здесь означает string. JSON `true` превратится в `True`, а `null` — в `None`, отсутствие значения. Посмотрим сначала весь результат, затем одно поле по его имени.


In [ ]:
record = json.loads(text)
print(type(record))
print(record)
print(record['name'])
print(record['experience'])


Не путайте отсутствие значения и ноль. `salary: null` означает «зарплата не указана», а `salary: 0` — указано число ноль. Если заменить все пропуски нулями, среднее станет отвечать на другой вопрос. Красивый график от этого не спасёт.

**Проверьте себя:** чем отличаются строка `"60000"` и число `60000`? Почему `"null"` в кавычках не превратится в `None`? Попробуйте изменить только одно значение в примере и заново выполнить обе ячейки.


### 2. Прочитаем одну страницу

`DATA_DIR / 'api' / ...` — путь к файлу. `read_text` читает текст, `encoding='utf-8'` задаёт его кодировку. Сначала получим строку, а уже потом разберём её: так видно, на каком шаге возникла бы ошибка. Метод словаря `.keys()` покажет имена полей результата.


In [ ]:
page_path = DATA_DIR / 'api' / 'vacancies-page-0.json'
page_text = page_path.read_text(encoding='utf-8')
page = json.loads(page_text)
print(page.keys())


`keys()` показывает имена полей словаря. Здесь `items` — записи текущей страницы, `page` — номер страницы, `pages` — количество страниц, `per_page` — размер страницы, `found` — общее количество записей в учебном наборе. Нумерация страниц начинается с нуля: при двух страницах нужны номера 0 и 1. У другого API названия и правила могут быть другими: смотрим документацию, не угадываем.


Возьмём `items`. Это список. `len` считает его элементы, `[0]` берёт первый. Следующая запись уже словарь: у неё есть поля `name`, `area`, `salary`. Разберём каждый уровень отдельно.


In [ ]:
items = page['items']
print(type(items))
print(len(items))
first = items[0]
print(first)


`area` — ещё один словарь. В нём поле `name` означает город. Одинаковое имя поля на разных уровнях не делает значения одинаковыми: `first['name']` — название позиции, `area['name']` — город. Сначала дадим промежуточным значениям имена, чтобы не потеряться в скобках.


In [ ]:
area = first['area']
city = area['name']
salary = first['salary']
lower = salary['from']
print(city)
print(lower)


### ✏️ Ваш ход — прочитать вложенный ответ

Создайте `page_count` — количество записей **этой** страницы, `first_city` — город первой записи и `first_lower` — нижнюю границу её зарплаты. Не вписывайте готовые числа: получите их из `page`. В выводе объясните, почему `found` не равно длине `items`.


In [ ]:
# ✏️ ваш код здесь


In [ ]:
check('04.1.1', page_count, first_city, first_lower)


**Мой вывод:** замените эту строку своим объяснением результата. Назовите конкретное число, поле или участок изображения, на который опираетесь.


### 3. А если зарплаты нет?

У второй записи `salary` равна `None`. Искать `['from']` внутри `None` нельзя: словаря там нет. Сначала проверяем значение. `is None` — проверка отсутствия; `if` выбирает ветку, `else` — другой случай. Обе ветки задают переменную `lower`, поэтому ниже её можно использовать.


In [ ]:
second = items[1]
salary = second['salary']
if salary is None:
    lower = None
else:
    lower = salary['from']
print(lower)


Но бывает, что **самого ключа** нет. У словаря метод `get('ключ')` возвращает `None`, если ключ отсутствует. Это не универсальное лечение ошибок: опечатка в имени поля тоже превратится в пропуск. Схему данных всё равно надо проверить. Попробуем на маленьком словаре, затем используем для необязательных полей.


In [ ]:
small = {'from': 60000}
print(small.get('from'))
print(small.get('to'))


### 4. Из одной записи сделаем одну строку

Пока без цикла. В новой строке сохраним ID, название, город и все признаки зарплаты: границы, валюту, признак `gross`. Здесь `gross=True` — сумма до налогов, `False` — после. `source_page` сохранит номер исходной страницы. Вместо отсутствующего словаря используем пустой `{}`: его `get` вернёт пропуски, а не выдуманные нули.


In [ ]:
item = items[0]
area = item['area']
salary = item.get('salary')
if salary is None:
    salary = {}
row = {
    'id': item['id'],
    'name': item['name'],
    'city': area['name'],
    'salary_from': salary.get('from'),
    'salary_to': salary.get('to'),
    'currency': salary.get('currency'),
    'gross': salary.get('gross'),
    'source_page': page['page']
}
print(row)


Теперь повторим **то же действие** для каждой записи. Список `rows` сначала пустой. Цикл `for item in items` по очереди кладёт в `item` одну запись, `append` добавляет готовую строку в список. Отступы показывают тело цикла. `DataFrame(rows)` превращает список словарей в таблицу: ключи станут столбцами.


In [ ]:
rows = []
for item in items:
    area = item['area']
    salary = item.get('salary')
    if salary is None:
        salary = {}
    row = {
        'id': item['id'], 'name': item['name'], 'city': area['name'],
        'salary_from': salary.get('from'), 'salary_to': salary.get('to'),
        'currency': salary.get('currency'), 'gross': salary.get('gross'),
        'source_page': page['page']
    }
    rows.append(row)
first_table = pd.DataFrame(rows)
first_table


В числовом столбце пропуск может отображаться как `NaN`: это представление отсутствующего значения в Pandas, а не текст «NaN» из исходного файла. Исходное поле мы не меняли. Сверьте первую и вторую строки с JSON.

**Пауза на опыт:** замените `items` на `items[:2]` в отдельной копии цикла. Срез `[:2]` берёт первые два элемента. Сколько строк получится? Затем верните исходный список. Не запускайте старую ячейку добавления к уже заполненному `rows`: инициализация `rows = []` должна происходить при каждом сборе заново.


### 5. Одной страницы мало

Прочитаем вторую страницу теми же двумя действиями. Обратите внимание на `page`: состав полей тот же, номер другой. Для сетевого API этот шаг был бы новым запросом с новым номером страницы, но здесь мы читаем локальный учебный файл.


In [ ]:
second_path = DATA_DIR / 'api' / 'vacancies-page-1.json'
second_text = second_path.read_text(encoding='utf-8')
second_page = json.loads(second_text)
print(second_page['page'])
print(len(second_page['items']))
pages = [page, second_page]


В `pages` лежат два словаря. Значит, теперь нужны два цикла: внешний берёт одну страницу (`for current_page in pages`), внутренний — одну запись этой страницы (`for item in current_page['items']`). После этого идёт уже знакомое создание строки. `rows = []` должно быть **перед обоими** циклами, иначе на каждой странице мы потеряем предыдущую.

Не будем прятать эти операции в сложную функцию: сначала важно увидеть, на каком уровне мы находимся. В реальном API, кроме страниц, могут быть курсоры или ссылка на следующую порцию; этот пример не является универсальным загрузчиком любого сайта.


### ✏️ Ваш ход — собрать обе страницы

Соберите таблицу `vacancies` из `pages`. Поля оставьте такими же, как в `first_table`; `source_page` берите у текущей страницы. Проверьте глазами запись без зарплаты, запись в USD и запись до налогов. Сколько получилось уникальных ID? Для проверки уникальности у столбца есть свойство `is_unique`: `vacancies['id'].is_unique`.


In [ ]:
# ✏️ ваш код здесь


In [ ]:
check('04.1.2', vacancies)


**Мой вывод:** замените эту строку своим объяснением результата. Назовите конкретное число, поле или участок изображения, на который опираетесь.


### 6. «Средняя зарплата» — звучит просто, но есть нюанс

Складывать рубли и доллары бессмысленно. Смешивать суммы до и после налогов тоже не стоит. Нижняя граница диапазона — не фактическая выплата человеку. Начнём с одного условия: `==` сравнивает значения, результат — ряд `True/False`, то есть маска строк.


In [ ]:
rub_mask = vacancies['currency'] == 'RUB'
print(rub_mask)
rub = vacancies[rub_mask]
rub


Отфильтруем ещё раз, теперь оставим суммы после налогов. Несколько простых шагов сейчас читаются лучше одной длинной строки. Наконец, `notna()` отметит известные нижние границы: объявление только с верхней границей в наш расчёт не подходит.


In [ ]:
net_mask = rub['gross'] == False
net = rub[net_mask]
known_mask = net['salary_from'].notna()
comparable = net[known_mask]
comparable[['id', 'salary_from', 'salary_to', 'currency', 'gross']]


Медиана — середина отсортированного ряда; при чётном количестве берём среднее двух центральных чисел. У Pandas метод столбца `.median()` делает это за нас. Перед расчётом надо назвать, **медиану чего** считаем: здесь нижних границ учебных предложений в рублях после налогов. По восьми придуманным записям нельзя делать вывод о рынке труда. И даже настоящий API не превращает удобную выборку в представительную автоматически.


### ✏️ Ваш ход — сформулировать честный вывод

В `comparable_count` сохраните количество сопоставимых записей; в `lower_median` — медиану их нижних границ. Напишите одну фразу с числом, единицей и ограничением. Затем подумайте: почему нельзя назвать это «столько получает начинающий специалист»?


In [ ]:
# ✏️ ваш код здесь


In [ ]:
check('04.1.3', comparable_count, lower_median)


**Мой вывод:** замените эту строку своим объяснением результата. Назовите конкретное число, поле или участок изображения, на который опираетесь.


### 7. Где здесь настоящий HTTP-запрос? Дополнительно

В основной части мы изучали структуру данных, а не обращались к серверу. В реальном API перед `json.loads` есть транспорт: отправить запрос, дождаться ответа, проверить его статус и формат. Например, документация GitHub описывает публичный метод получения сведений о репозитории. Учётная запись для этого публичного чтения не обязательна; доступ и лимиты определяет сервис.

Ниже **пример для чтения**, а не обязательная ячейка. `requests.get` отправляет GET-запрос. `timeout=10` ограничивает ожидание подключения/чтения; это не обещание точного общего времени. `status_code` — код HTTP; `raise_for_status()` останавливает выполнение при HTTP-ошибке. `response.json()` разбирает тело ответа. Даже успешный статус не гарантирует нужную схему: сервис может изменить поля или прислать другой документ.

```python
import requests
url = 'https://api.github.com/repos/python/cpython'
response = requests.get(url, timeout=10)
print(response.status_code)
response.raise_for_status()
data = response.json()
print(data['full_name'])
```

В блоке 03.1 уже были сетевые ошибки. Здесь порядок тот же: сначала проверьте сеть/статус, затем формат, затем поля. При отказе доступа не повторяйте запрос бесконечно и не пытайтесь обходить ограничения. Для проекта запишите адрес, параметры, дату получения и правила использования. Токены в тетрадь и публичный репозиторий не помещают. Если источник требует авторизацию, для этой пары достаточно локального примера.

Источники: [модуль json](https://docs.python.org/3/library/json.html), [GitHub REST: Get a repository](https://docs.github.com/en/rest/repos/repos#get-a-repository), [книга Уэса Маккинни](https://wesmckinney.com/book/).

### Что сохраняем

Сохраните тетрадь с тремя ответами и собственными выводами. В следующей паре разберём случай, когда данные отдают сразу файлом: там тоже бывают сюрпризы, только вместо HTML — кодировки и разделители. Для проекта выберите один источник, который реально сможете получить повторно; инструкция ЛР2 доступна в `docs/ЛР2.md`.
